# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos bsuscados.

In [2]:
from bs4 import BeautifulSoup

file = 'rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [3]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [4]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [7]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [14]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

recipe_urls = set()

for link in recipe_links:

    href = link.get("href", "")

    if (
        "/recipe/" in href
        and (
            href.startswith("/")
            or "allrecipes.com" in href
        )
    ):
        recipe_urls.add(href)

recipe_urls = sorted(recipe_urls)

print(f"Recetas encontradas: {len(recipe_urls)}")

Recetas encontradas: 16


## Parte 4: Descargar recetas de las URL

In [15]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://www.allrecipes.com"
recipe_urls = list(set(recipe_urls))

full_urls = []

for url in recipe_urls:
    if url.startswith("http"):
        full_urls.append(url)
    else:
        full_urls.append(urljoin(BASE_URL, url))

print(f"Total recetas encontradas: {len(full_urls)}")

Total recetas encontradas: 16


## Parte 5: Extracción de recetas

In [ ]:
import requests
from bs4 import BeautifulSoup

# Crear una sesión reutilizable
session = requests.Session()

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
})


def extract_recipe(url):

    try:
        response = session.get(url, timeout=30)

        if response.status_code != 200:
            print(f"Error {response.status_code}: {url}")
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        title_tag = soup.find("meta", {"property": "og:title"})
        title = title_tag["content"] if title_tag else ""

        description_tag = soup.find("meta", {"name": "description"})
        description = description_tag["content"] if description_tag else ""
        ingredients = [
            ingredient.get_text(strip=True)
            for ingredient in soup.find_all(
                "li",
                class_="mm-recipes-structured-ingredients__list-item"
            )
        ]
        instructions = [
            instruction.get_text(strip=True)
            for instruction in soup.find_all(
                "p",
                class_="comp mntl-sc-block mntl-sc-block-html"
            )
        ]
        nutrition = [
            fact.parent.get_text(" ", strip=True)
            for fact in soup.find_all(
                "span",
                class_="mm-recipes-nutrition-facts-label__nutrient-name "
                      "mm-recipes-nutrition-facts-label__nutrient-name--has-postfix"
            )
        ]

        return {
            "url": url,
            "title": title,
            "description": description,
            "ingredients": ingredients,
            "instructions": instructions,
            "nutrition": nutrition
        }

    except requests.exceptions.RequestException as e:
        print(f"Error de conexión con {url}")
        print(e)
        return None

    except Exception as e:
        print(f"Error procesando {url}")
        print(e)
        return None

Recorrer todas las recetas usando la función:

In [22]:
recipes = []

for url in full_urls:

    print("Procesando:", url)

    recipe = extract_recipe(url)

    if recipe is not None:
        recipes.append(recipe)

print(f"\nRecetas extraídas: {len(recipes)}")

Procesando: https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
Procesando: https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
Procesando: https://www.allrecipes.com/recipe/214618/beer-can-chicken/
Procesando: https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
Procesando: https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
Procesando: https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
Procesando: https://www.allrecipes.com/recipe/19944/drunk-chicken/
Procesando: https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
Procesando: https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
Procesando: https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
Procesando: https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
Procesando: https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
Procesando: https://www.allrecipes.com/recipe/34957/easy-barbeque-chicken/


## Parte 6: Data frame

In [45]:
df = pd.DataFrame(recipes)

df.head()
df

,url,title,description,ingredients,instructions,nutrition
0,https://www.allrecipes.com/recipe/258659/rosem...,Rosemary Buttermilk Chicken,Summer is the time to master this garlicky but...,"[2 ½cupsbuttermilk, 15clovesgarlic, minced, 1t...","[Mix buttermilk, garlic, paprika, salt, and pe...","[Total Carbohydrate 8g, Dietary Fiber 1g, Tota..."
1,https://www.allrecipes.com/recipe/14531/beer-b...,Beer Butt Chicken,"This beer butter chicken recipe combines beer,...","[1cupbutter, divided, 2tablespoonsgarlic salt,...",[Preheat an outdoor grill for low heat and lig...,"[Total Carbohydrate 3g, Dietary Fiber 1g, Tota..."
2,https://www.allrecipes.com/recipe/214618/beer-...,Beer Can Chicken,This beer can chicken cooks a perfectly season...,"[⅓cupbrown sugar, 2tablespoonschili powder, 2t...",[Preheat a charcoal grill for medium-high heat...,"[Total Carbohydrate 24g, Dietary Fiber 3g, Tot..."
3,https://www.allrecipes.com/recipe/281255/smoke...,Smoked Whole Chicken,Spatchcocked and rubbed with a spicy seasoning...,"[2tablespoonspaprika, 2tablespoonschili powder...",[Preheat a smoker to 250 degrees F (120 degree...,"[Total Carbohydrate 8g, Dietary Fiber 3g, Tota..."
4,https://www.allrecipes.com/recipe/222936/smoke...,Smoked Beer Butt Chicken,Roast a chicken on your grill with a can of be...,"[1(3 pound)whole chicken, ¼cupvegetable oil, 3...",[Preheat grill for medium heat and lightly oil...,"[Total Carbohydrate 10g, Dietary Fiber 1g, Tot..."
5,https://www.allrecipes.com/recipe/274724/grill...,Grilled Spatchcocked Chicken,This grilled spatchcock chicken recipe calls f...,"[¼cupkosher salt, water, 1(4 pound)whole chick...",[Place salt in a large bowl or Dutch oven; add...,"[Total Carbohydrate 5g, Dietary Fiber 1g, Tota..."
6,https://www.allrecipes.com/recipe/19944/drunk-...,Drunk Chicken,Drunken chicken is cooked over the grill with ...,"[1(2 to 3 pound)whole chicken, 1(12 fluid ounc...",[Rinse and dry chicken. Remove excess fat and ...,"[Total Carbohydrate 6g, Dietary Fiber 1g, Tota..."
7,https://www.allrecipes.com/recipe/275062/butte...,Buttermilk Barbecue Chicken,Chicken turns out super moist and flavorful on...,"[2cupsbuttermilk, ¼cupbrown sugar, 1tablespoon...","[Whisk buttermilk, brown sugar, cider vinegar,...","[Total Carbohydrate 15g, Dietary Fiber 1g, Tot..."
8,https://www.allrecipes.com/recipe/214619/bbq-b...,Best Beer Can Chicken,"In this best beer can chicken recipe, chicken ...","[2cupscherry wood chips, ½cupdark brown sugar,...",[Soak wood chips in water for at least 1 hour....,"[Total Carbohydrate 23g, Dietary Fiber 4g, Tot..."
9,https://www.allrecipes.com/recipe/264278/miso-...,Miso Honey Chicken,Chicken is marinated in Chef John's magical mi...,"[3tablespoonswhite miso, 2tablespoonshoney, ¼c...",[Combine miso and honey in a bowl. Pour in ric...,"[Total Carbohydrate 7g, Dietary Fiber 1g, Tota..."


Creación del Corpus:

In [28]:
def build_document(row):

    ingredients = "\n".join(f"- {x}" for x in row["ingredients"])

    instructions = "\n".join(
        f"{i+1}. {step}"
        for i, step in enumerate(row["instructions"])
    )

    nutrition = "\n".join(row["nutrition"])

    document = f"""
    Recipe: {row['title']}
    Description:
    {row['description']}
    Ingredients:
    {ingredients}
    Instructions:
    {instructions}
    Nutrition:
    {nutrition}
    """
    return document

In [29]:
df["document"] = df.apply(build_document, axis=1)

df["document"].iloc[0]

'\n    Recipe: Rosemary Buttermilk Chicken\n    Description:\n    Summer is the time to master this garlicky buttermilk chicken grilling recipe that is simply infused with fresh rosemary and smoky paprika.\n    Ingredients:\n    - 2 ½cupsbuttermilk\n- 15clovesgarlic, minced\n- 1tablespoonsmoked paprika\n- kosher salt and ground black pepper to taste\n- 8sprigsfresh rosemary\n- 1(3 to 3 1/2 pound)whole chicken, quartered\n    Instructions:\n    1. Mix buttermilk, garlic, paprika, salt, and pepper together in a large bowl; stir to combine.\n2. Massage rosemary sprigs to release fragrant oils. Put rosemary in a large resealable plastic bag; pour in buttermilk mixture. Add chicken parts to the bag, seal carefully, and place in refrigerator. Marinate, turning the bag occasionally, 4 hours to overnight.\n3. Preheat grill for medium heat and lightly oil the grate.\n4. Transfer chicken to a plate; discard marinade. Grill chicken until no longer pink at the bone and the juices run clear, turnin

## Parte 7: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

In [48]:
from langchain_core.documents import Document

documents = []

for i, (_, row) in enumerate(df.iterrows()):

    doc = Document(
        page_content=build_document(row),
        metadata={
            "title": row["title"],
            "url": row["url"],
            "recipe_id": i
        }
    )

    documents.append(doc)

print(f"Documentos creados: {len(documents)}")

Documentos creados: 16


In [49]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

splits = text_splitter.split_documents(documents)

print(f"Chunks creados: {len(splits)}")

splits = text_splitter.split_documents(documents)

print(f"Chunks creados: {len(splits)}")

Chunks creados: 74
Chunks creados: 74


Embeddings

In [50]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [
    doc.page_content
    for doc in splits
]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print(embeddings.shape)

Batches: 100%|██████████| 3/3 [00:03<00:00,  1.01s/it]

(74, 384)


## Parte 8: Base de datos (Faiss)

In [51]:
import faiss
import numpy as np

embedding_matrix = np.array(embeddings).astype("float32")

dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embedding_matrix)

print("Vectores almacenados:", index.ntotal)

Vectores almacenados: 74


## Busqueda:

In [52]:
def search_recipe(query, k=3):

    query_embedding = embedding_model.encode(
        [query]
    )

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k
    )

    # Mejor coincidencia
    best_chunk = splits[indices[0][0]]

    recipe_id = best_chunk.metadata["recipe_id"]

    # Recuperar todos los chunks de esa receta
    full_recipe_chunks = [
        doc for doc in splits
        if doc.metadata["recipe_id"] == recipe_id
    ]

    print("=" * 80)
    print("RECETA CON MAYOR COINCIDENCIA")
    print("=" * 80)

    print("Título:")
    print(best_chunk.metadata["title"])

    print("\nURL:")
    print(best_chunk.metadata["url"])

    print("\nReceta completa:")
    
    for chunk in full_recipe_chunks:
        print(chunk.page_content)
        print()


    print("\n" + "=" * 80)
    print("OTRAS RECETAS RELACIONADAS")
    print("=" * 80)

    shown = set()

    for idx in indices[0]:

        recipe = splits[idx]

        title = recipe.metadata["title"]

        if (
            title != best_chunk.metadata["title"]
            and title not in shown
        ):
            print("-", title)
            shown.add(title)

    return full_recipe_chunks

In [53]:
result = search_recipe(
    "How can I grill a whole chicken?"
)

RECETA CON MAYOR COINCIDENCIA
Título:
Beer Can Chicken

URL:
https://www.allrecipes.com/recipe/214618/beer-can-chicken/

Receta completa:
Recipe: Beer Can Chicken
    Description:
    This beer can chicken cooks a perfectly seasoned whole chicken on a grill until it's deliciously crisp and moist, full of flavor, and easy to carve.
    Ingredients:
    - ⅓cupbrown sugar
- 2tablespoonschili powder
- 2tablespoonspaprika
- 2teaspoonsdry mustard
- ½teaspoonsalt
- ¼teaspoonground black pepper
- ½(12 fluid ounce) canbeer
- 1(3 pound)whole chicken
    Instructions:

- 1(3 pound)whole chicken
    Instructions:
    1. Preheat a charcoal grill for medium-high heat, about 375 degrees F (190 degrees C).
2. Mix brown sugar, chili powder, paprika, dry mustard, salt, and black pepper in a small bowl. Place half-full can of beer in the center of a plate.

3. Fit whole chicken over the can of beer with the legs on the bottom; keep upright. Sprinkle 1 teaspoon of seasoning mix into the top cavity of chic